# 02 — CV Split Generation

Build 5-fold CSVs for one (DATASET, SPLIT_SCHEME) combination and write them under:

    data/<dataset>/splits/<scheme>/folds/fold_X_{train,val,test}.csv
    data/<dataset>/splits/<scheme>/cv_split_config.json
    data/<dataset>/splits/<scheme>/cv_fold_summary.csv

Two split schemes:

- **`patient_level`** — `StratifiedGroupKFold` on `patient_id`. Patients are guaranteed disjoint across train/val/test. **Use for final results.**
- **`image_level`** — plain `KFold` on rows (images). Patients can leak across folds. Used only to reproduce the FigShare reference benchmark's leaky-split numbers.

**Run this notebook 4 times** (once per combination):

| DATASET    | SPLIT_SCHEME   |
|------------|----------------|
| figshare   | patient_level  |
| figshare   | image_level    |
| brats2020  | patient_level  |
| brats2020  | image_level    |

Each run only touches its own scheme directory and does not clobber others.

NB02 writes **directly to Drive** (small CSVs, not in the heavy-write training hot path).

## Cell 1 — Install dependencies

In [ ]:
%pip install -q pandas scikit-learn numpy

## Cell 2 — Bootstrap: mount Drive + clone/pull repo

In [ ]:
import os, sys

if not os.path.exists("/content/senior_project"):
    from google.colab import userdata
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    url = "https://github.com/salemaker47/senior_project.git"
    if token:
        url = url.replace("https://", f"https://{token}@", 1)
    os.system(f"git clone {url} /content/senior_project")
if "/content/senior_project" not in sys.path:
    sys.path.insert(0, "/content/senior_project")

from src.notebook_setup import setup_environment

DRIVE_ROOT, REPO_ROOT = setup_environment(
    repo_url="https://github.com/salemaker47/senior_project.git",
)

# NB02 exception: write directly to Drive (small CSVs).
PROJECT_ROOT = DRIVE_ROOT

print(f"DRIVE_ROOT:   {DRIVE_ROOT}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

## Cell 3 — Knobs

Change `DATASET` and `SPLIT_SCHEME`, then re-run all cells for each combination.

In [ ]:
DATASET      = "figshare"         # one of: "figshare", "brats2020"
SPLIT_SCHEME = "patient_level"    # one of: "patient_level", "image_level"

N_SPLITS     = 5
VAL_RATIO    = 0.15    # fraction of the train pool held out for validation
RANDOM_SEED  = 42

assert DATASET in ("figshare", "brats2020"), f"unknown DATASET: {DATASET!r}"
assert SPLIT_SCHEME in ("patient_level", "image_level"), f"unknown SPLIT_SCHEME: {SPLIT_SCHEME!r}"

print(f"DATASET:      {DATASET}")
print(f"SPLIT_SCHEME: {SPLIT_SCHEME}")
print(f"N_SPLITS:     {N_SPLITS}")
print(f"VAL_RATIO:    {VAL_RATIO}")
print(f"RANDOM_SEED:  {RANDOM_SEED}")

## Cell 4 — Load and validate metadata.csv

In [ ]:
from src.file_utils import dataset_paths, split_scheme_dir
from src.data_utils import load_metadata, validate_metadata, metadata_summary

paths = dataset_paths(PROJECT_ROOT, DATASET)
metadata_df = load_metadata(paths["metadata_csv"])
print(f"metadata_df shape: {metadata_df.shape}")

validate_metadata(metadata_df)
print("validate_metadata: OK")

print()
print("Overall summary:")
print(metadata_summary(metadata_df).to_string(index=False))

## Cell 5 — Generate K outer folds

In [ ]:
from src.data_utils import (
    create_patient_folds, create_image_level_folds,
)

if SPLIT_SCHEME == "patient_level":
    splits, splitter_used = create_patient_folds(
        metadata_df, n_splits=N_SPLITS, random_state=RANDOM_SEED,
    )
else:
    splits, splitter_used = create_image_level_folds(
        metadata_df, n_splits=N_SPLITS, random_state=RANDOM_SEED,
    )

print(f"splitter: {splitter_used}")
print(f"n_folds:  {len(splits)}")

## Cell 6 — Build per-fold DataFrames

For each outer fold:
1. Split metadata into `train_pool` (4/5) and `test` (1/5).
2. Carve `val` off `train_pool` (per-scheme split logic).
3. (Patient-level only) Hard-assert no patient leakage across train/val/test.


In [ ]:
from src.data_utils import (
    make_train_val_from_pool, make_train_val_image_level,
    verify_no_patient_leakage,
)

fold_dfs = {}        # {fold: {"train": df, "val": df, "test": df}}
fold_counts = []     # rows for cv_fold_summary.csv

for k, (pool_idx, test_idx) in enumerate(splits, start=1):
    pool_df = metadata_df.iloc[pool_idx]
    test_df = metadata_df.iloc[test_idx].reset_index(drop=True)

    if SPLIT_SCHEME == "patient_level":
        train_df, val_df = make_train_val_from_pool(
            pool_df, val_ratio=VAL_RATIO, random_state=RANDOM_SEED,
        )
        leak = verify_no_patient_leakage(train_df, val_df, test_df)
        print(f"fold {k}: {leak}")
    else:
        train_df, val_df = make_train_val_image_level(
            pool_df, val_ratio=VAL_RATIO, random_state=RANDOM_SEED,
        )
        print(f"fold {k}: train={len(train_df)} val={len(val_df)} test={len(test_df)}")

    fold_dfs[k] = {"train": train_df, "val": val_df, "test": test_df}

    for split_name, df in fold_dfs[k].items():
        row = {
            "fold": k,
            "split": split_name,
            "n_images": int(len(df)),
            "n_patients": int(df["patient_id"].nunique()),
        }
        for cls, count in df["tumor_class"].value_counts().items():
            row[f"n_class_{cls}"] = int(count)
        fold_counts.append(row)

## Cell 7 — Persist fold CSVs and config

In [ ]:
from datetime import datetime, timezone
import pandas as pd

from src.data_utils import save_fold_csvs
from src.file_utils import save_json

scheme_dir = split_scheme_dir(PROJECT_ROOT, DATASET, SPLIT_SCHEME)
folds_dir  = scheme_dir / "folds"

# 1. Fold CSVs (15 total: 5 folds x train/val/test)
saved = save_fold_csvs(fold_dfs, folds_dir)
print(f"Wrote fold CSVs under {folds_dir}")
for fold_num, parts in saved.items():
    for split_name, p in parts.items():
        print(f"  fold {fold_num} {split_name}: {p.relative_to(PROJECT_ROOT)}")

# 2. cv_split_config.json — recipe for reproducibility
config = {
    "dataset":          DATASET,
    "split_scheme":     SPLIT_SCHEME,
    "splitter_used":    splitter_used,
    "n_splits":         N_SPLITS,
    "val_ratio":        VAL_RATIO,
    "random_state":     RANDOM_SEED,
    "n_metadata_rows":  int(len(metadata_df)),
    "n_metadata_patients": int(metadata_df["patient_id"].nunique()),
    "generated_at":     datetime.now(timezone.utc).isoformat(),
}
config_path = scheme_dir / "cv_split_config.json"
save_json(config, config_path)
print(f"\nWrote {config_path.relative_to(PROJECT_ROOT)}")

# 3. cv_fold_summary.csv — per-(fold, split) counts
summary_df = pd.DataFrame(fold_counts).fillna(0)
# stable column order
front = ["fold", "split", "n_images", "n_patients"]
class_cols = sorted(c for c in summary_df.columns if c.startswith("n_class_"))
summary_df = summary_df[front + class_cols]
for c in ["n_images", "n_patients"] + class_cols:
    summary_df[c] = summary_df[c].astype(int)

summary_path = scheme_dir / "cv_fold_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Wrote {summary_path.relative_to(PROJECT_ROOT)}")

print()
print("Per-fold summary:")
print(summary_df.to_string(index=False))

## Done.

Drive now contains, for the chosen (DATASET, SPLIT_SCHEME) combination:

```
data/<DATASET>/splits/<SCHEME>/
├── folds/fold_X_{train,val,test}.csv   (15 files: 5 folds × 3 splits)
├── cv_split_config.json
└── cv_fold_summary.csv
```

**Still needed:** run this notebook for the remaining 3 combinations:

| DATASET    | SPLIT_SCHEME   | Status |
|------------|----------------|--------|
| figshare   | patient_level  |        |
| figshare   | image_level    |        |
| brats2020  | patient_level  |        |
| brats2020  | image_level    |        |

Next: **Segmentation training** (`03_train.ipynb`).